In [12]:
 
import json
import linecache 
import pandas as pd
import time 
import numpy as np
import matplotlib.pyplot as plt


In [13]:

class CsvConverter:
    """
    This is a class for converting a csv file into a json file. 
    
    Attributes:
        csvfile (str): The name of the csv file to be converted.
        jsonfile (str): The name of the json file to be created.
        keys (list): The list of keys (header) in the csv file.
        data (list): The list of data rows in the csv file.
    """

    def __init__(self, csvfile, jsonfile):
        """
        The constructor for the CsvConverter class.
        
        Parameters:
           csvfile (str): The name of the csv file to be converted.
           jsonfile (str): The name of the json file to be created.
        """
        self.csvfile = csvfile
        self.jsonfile = jsonfile
        self.keys = self.header_maker()  # creating the list of keys from the csv header
        self.data = self.data_maker()  # creating the list of data rows from the csv file
    
    
    def header_maker(self):
        """
        The function to extract header (keys) from the csv file.

        Returns:
            list: The list of keys in the csv file.
        """
        header_line = linecache.getline(self.csvfile, 1)
        keys = header_line.strip().split(',')
        return keys
 
    def data_maker(self):
        """
        The function to extract data from the csv file.

        Returns:
            list: The list of data rows in the csv file.
        """
 
        data = []
        rows = len(linecache.getlines(self.csvfile))  # calculating the number of lines in the csv file
        for i in range(2, rows+1):  # iterating over the lines of the csv file
            row_line = linecache.getline(self.csvfile, i)  # reading a line from the csv file
            row_values = row_line.strip().split(',')  # splitting the line into a list of values
            # checking if the number of values in the line matches the number of keys in the header
            assert len(row_values) == len(self.keys), f"Warning: Skipping line {i-1} as number of elements don't match number of keys in header"
            row_dict = dict(zip(self.keys, row_values))  # creating a dictionary from the list of keys and the list of values
            data.append(row_dict)  # adding the dictionary to the list of data rows
    
        return data    

 
    def csv_to_json(self):
        """
        The function to convert the csv file into a json file.
        """
 
        json_data = {'data': self.data, 'header': self.keys}  # creating a dictionary from the list of keys and the list of data rows
        with open(self.jsonfile, 'w') as json_file:  # opening the json file in write mode
            json.dump(json_data, json_file, indent=4)  # dumping the dictionary into the json file


In [ ]:
converter = CsvConverter('dSST.csv', 'output.json') 

In [14]:
class Reader:
    """
    Reader class for reading data from a CSV file.

    This class provides a reader that can read data from a CSV file in a stride manner
    and notifies its observers every time new data is read. 
    
    Attributes:
        csvfile: A string indicating the path to the csv file.
        stride: An integer indicating the number of lines to be read in each read operation.
        pointer: An integer indicating the current position in the file.
        converter: An instance of CsvConverter that converts the CSV data to JSON format.
        data: A list of dictionaries representing the CSV data.
        keys: A list of strings representing the keys (columns) in the CSV data.
        observers: A set of observers to be notified when new data is read.

    Methods:
        add_observer(observer): Adds an observer to the set of observers.
        remove_observer(observer): Removes an observer from the set of observers.
        notify_observers(): Notifies all observers that new data has been read.
        get_lines(): Reads the next stride of lines from the data, notifies the observers, 
                     and returns the data in JSON format.
    """

    def __init__(self, csvfile, stride):
        """
        Initializes a new instance of the Reader class.

        Args:
            csvfile: A string indicating the path to the csv file.
            stride: An integer indicating the number of lines to be read in each read operation.
        """
        self.csvfile = csvfile
        self.stride = stride
        self.pointer = 0
        self.converter = CsvConverter(self.csvfile, 'foo.json')
        self.data = self.converter.data
        self.keys = self.converter.keys
        self.observers = set()

    def add_observer(self, observer):
        """
        Adds an observer to the set of observers.

        Args:
            observer: The observer to be added.
        """
        self.observers.add(observer)

    def remove_observer(self, observer):
        """
        Removes an observer from the set of observers.

        Args:
            observer: The observer to be removed.
        """
        self.observers.discard(observer)

    def notify_observers(self):
        """
        Notifies all observers that new data has been read.
        """
        for observer in self.observers:
            observer.update()

    def get_lines(self):
        """
        Reads the next stride of lines from the data, notifies the observers, 
        and returns the data in JSON format.

        Returns:
            A string containing the data in JSON format.
        """
        if self.pointer >= len(self.data):
            return ''
        lines = self.data[self.pointer:self.pointer+self.stride]
        self.pointer += self.stride
        json_data = {'data': lines}
        time.sleep(5) 
        self.notify_observers()
        return json.dumps(json_data, indent = 4)


In [ ]:
reader = Reader('dSST.csv', 5)
while True:
    lines = reader.get_lines()
    if not lines:
        break
    print(lines)


In [15]:
class AverageYear:
    """
    The AverageYear class processes a CSV file with climate data and calculates the average temperature anomaly for each year.
    The results are then visualized with a line plot.

    Attributes:
    -----------
    reader : Reader
        A Reader object that reads data from a CSV file.
    data : pd.DataFrame
        A pandas DataFrame storing the data read from the CSV file.
    fig, ax : matplotlib.pyplot.subplots
        Matplotlib objects for creating the line plot.
    """

    def __init__(self, csvfile, stride):
        """
        Constructs all the necessary attributes for the AverageYear object.

        Parameters:
        ----------
        csvfile : str
            Path to the CSV file to read.
        stride : int
            The number of lines to read at a time from the CSV file.
        """
        self.reader = Reader(csvfile, stride)
        self.data = pd.DataFrame()
        self.fig, self.ax = plt.subplots()
        self.reader.add_observer(self)
    
    def calculate_average(self):
        """
        Reads data from the CSV file, calculates the average temperature anomaly for each year, 
        and creates a line plot of the results.
        """
        # Read the lines from the CSV file and convert to DataFrame
        while True:
            lines = self.reader.get_lines()
            if not lines:
                break
            json_data = json.loads(lines)
            df = pd.DataFrame(json_data['data'])
            df = df.astype({'Year': int})
            df = df.set_index('Year')
            self.data = self.data.append(df)
            
        # Convert all temperature anomaly columns to numeric
        self.data[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']] = self.data[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']].apply(pd.to_numeric)
        
        # Calculate average temperature anomaly for each year
        self.data['Yearly Average'] = self.data[['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']].mean(axis=1)
        
        # Create a line plot of the average temperature anomaly for each year
        self.ax.plot(self.data.index, self.data['Yearly Average'])
        self.ax.set_xlabel('Year')
        self.ax.set_ylabel('Average Temperature Anomaly')
        self.ax.set_title('Average Temperature Anomaly for All Years')
        plt.show()

    def update(self):
        """
        Calls the calculate_average method when new data is available from the Reader.
        """
        self.calculate_average()    


In [ ]:
average_year = AverageYear('dSST.csv',5)
average_year.calculate_average()

In [16]:
class AverageMonth:
    """
    The AverageMonth class processes a CSV file with climate data and calculates the average temperature anomaly for each month over all years.
    The results are then visualized with a line plot.

    Attributes:
    -----------
    reader : Reader
        A Reader object that reads data from a CSV file.
    data : pd.DataFrame
        A pandas DataFrame storing the data read from the CSV file.
    fig, ax : matplotlib.pyplot.subplots
        Matplotlib objects for creating the line plot.
    months : list
        A list of string month abbreviations.
    """

    def __init__(self, csvfile, stride):
        """
        Constructs all the necessary attributes for the AverageMonth object.

        Parameters:
        ----------
        csvfile : str
            Path to the CSV file to read.
        stride : int
            The number of lines to read at a time from the CSV file.
        """
        self.reader = Reader(csvfile, stride)
        self.data = pd.DataFrame()
        self.fig, self.ax = plt.subplots()
        self.months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
        self.reader.add_observer(self)
    
    def calculate_average(self):
        """
        Reads data from the CSV file, calculates the average temperature anomaly for each month over all years,
        and creates a line plot of the results.
        """
        # Read lines from CSV file and convert to DataFrame
        while True:
            lines = self.reader.get_lines()
            if not lines:
                break
            json_data = json.loads(lines)
            df = pd.DataFrame(json_data['data'])
            df = df.astype({'Year': int})
            df = df[self.months] 
            self.data = self.data.append(df)
         
        # Convert all temperature anomaly columns to numeric and calculate average for each month
        self.data = self.data.apply(pd.to_numeric)
        self.monthly_averages = self.data.mean(axis=0)
 
        # Create a line plot of the average temperature anomaly for each month
        self.ax.plot(self.months, self.monthly_averages)
        self.ax.set_xlabel('Month')
        self.ax.set_ylabel('Average Monthly Temperature Anomaly')
        self.ax.set_title('Average Monthly Temperature Anomaly for All Years')
        plt.show()
        
    def update(self):
        """
        Calls the calculate_average method when new data is available from the Reader.
        """
        self.calculate_average()    


In [ ]:
average_month = AverageMonth('dSST.csv', 5)
average_month.calculate_average()
